In [4]:
import sys
sys.path.insert(0, '.')

from utils import load_frames, volume_preserving_distortion
import py3Dmol
from ase.io import write
import io

In [6]:
frames = load_frames('/home/yuejian/project/MLFF-distill/yuejian/electrolyte/test/train')
frame = frames[0]
print(f'Loaded frame: {len(frame)} atoms')
print(f'Cell before: {frame.get_cell().diagonal()}')

Loading: 100%|██████████| 10/10 [00:00<00:00, 115.17it/s]

Loaded frame: 2584 atoms
Cell before: [29.61336422 27.5746956  29.04195927]


In [27]:
stretched = volume_preserving_distortion(frame, max_stretch=0.99)
print(f'Cell before: {frame.get_cell().diagonal()}')
print(f'Cell after:  {stretched.get_cell().diagonal()}')
print(f'Volume before: {frame.get_volume():.3f} A3')
print(f'Volume after:  {stretched.get_volume():.3f} A3')

Cell before: [29.61336422 27.5746956  29.04195927]
Cell after:  [  3.4374822   49.3359419  139.83648715]
Volume before: 23715.069 A3
Volume after:  23715.069 A3


In [28]:
def atoms_to_xyz(atoms):
    buf = io.StringIO()
    write(buf, atoms, format='xyz')
    return buf.getvalue()

def show_atoms(atoms, title='', width=600, height=450):
    xyz = atoms_to_xyz(atoms)
    view = py3Dmol.view(width=width, height=height)
    view.addModel(xyz, 'xyz')
    view.setStyle({'sphere': {'radius': 0.3}, 'stick': {'radius': 0.1}})
    cell = atoms.get_cell()
    a, b, c = cell[0].tolist(), cell[1].tolist(), cell[2].tolist()
    o = [0.0, 0.0, 0.0]
    edges = [
        (o, a), (o, b), (o, c),
        (a, [a[i]+b[i] for i in range(3)]),
        (a, [a[i]+c[i] for i in range(3)]),
        (b, [a[i]+b[i] for i in range(3)]),
        (b, [b[i]+c[i] for i in range(3)]),
        (c, [a[i]+c[i] for i in range(3)]),
        (c, [b[i]+c[i] for i in range(3)]),
        ([a[i]+b[i] for i in range(3)], [a[i]+b[i]+c[i] for i in range(3)]),
        ([a[i]+c[i] for i in range(3)], [a[i]+b[i]+c[i] for i in range(3)]),
        ([b[i]+c[i] for i in range(3)], [a[i]+b[i]+c[i] for i in range(3)]),
    ]
    for s, e in edges:
        view.addCylinder({'start': {'x': s[0], 'y': s[1], 'z': s[2]},
                          'end':   {'x': e[0], 'y': e[1], 'z': e[2]},
                          'radius': 0.08, 'color': 'gray', 'opacity': 0.9})
    view.zoomTo()
    print(title)
    return view.show()

In [29]:
show_atoms(frame, title=f'Before — cell: {frame.get_cell().diagonal().round(3)} A')

Before — cell: [29.613 27.575 29.042] A


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [26]:
show_atoms(stretched, title=f'After  — cell: {stretched.get_cell().diagonal().round(3)} A')

After  — cell: [ 47.644   4.257 116.929] A


3Dmol.js failed to load for some reason. Please check your browser console for error messages.